# rankviz tutorial — why uniformity beats specialisation

This notebook demonstrates rankviz's three core visualisations using
structured synthetic data that reproduces a key finding from adversarial
retrieval research: a single document placed at the **geometric median**
of multiple query clusters outranks specialist documents that are much
closer to their home cluster.

Standard dimensionality reduction (PCA, t-SNE, UMAP) cannot reveal this
because they optimise for geometry, not ranking. rankviz shows retrieval
behaviour directly.

In [ ]:
import numpy as np
import rankviz
from rankviz import RankCarpet, SimilarityWaterfall, RankDistribution

print(f"rankviz {rankviz.__version__}")

## 1. Generate structured synthetic data

We create three well-separated query clusters on the unit sphere in 128-D,
each representing a different domain.  The corpus contains specialist
documents drawn near each cluster.  The "poison" document is placed at
the geometric median of the three cluster centres — moderately similar
to all queries, but a top specialist for none.

In [ ]:
def l2_normalise(x):
    """Row-wise L2 normalisation."""
    norms = np.linalg.norm(x, axis=-1, keepdims=True)
    return x / np.where(norms == 0, 1, norms)


rng = np.random.default_rng(2026)
dim = 128

# Three cluster centres, well separated on the unit sphere.
centres = l2_normalise(rng.standard_normal((3, dim)))

# Queries: 50 per domain, tight clusters.
domain_names = ["medicine", "finance", "legal"]
queries, query_labels = [], []
for i, (c, name) in enumerate(zip(centres, domain_names)):
    noise = rng.standard_normal((50, dim)) * 0.12
    cluster = l2_normalise(c + noise)
    queries.append(cluster)
    query_labels.extend([name] * 50)

queries = np.concatenate(queries).astype(np.float32)
query_labels = np.array(query_labels)

# Corpus: 200 specialists per domain (wider spread).
corpus = []
for c in centres:
    noise = rng.standard_normal((200, dim)) * 0.25
    docs = l2_normalise(c + noise)
    corpus.append(docs)
corpus = np.concatenate(corpus).astype(np.float32)

# Poison: geometric median of the three cluster centres.
poison = l2_normalise(centres.mean(axis=0, keepdims=True)).astype(np.float32).squeeze()

# Pick one specialist for comparison.
specialist = corpus[0]  # a medicine specialist

print(f"Queries:     {queries.shape}")
print(f"Corpus:      {corpus.shape}")
print(f"Poison:      {poison.shape}")
print(f"Domains:     {np.unique(query_labels).tolist()}")

### Sanity check: cosine similarities

The poison has moderate but uniform similarity to all queries.
The specialist has high similarity to medicine queries and low
similarity to the other two domains.

In [ ]:
sim_poison = queries @ poison
sim_specialist = queries @ specialist

for domain in domain_names:
    mask = query_labels == domain
    print(f"{domain:10s}  poison cos={sim_poison[mask].mean():.3f}  "
          f"specialist cos={sim_specialist[mask].mean():.3f}")

## 2. Rank Carpet

The Rank Carpet shows each document's rank across all queries.
The poison (red) maintains a uniformly low rank across all domains;
the specialist (green) has very low ranks in medicine but falls
far down the ranking in the other two domains.

Domain separators (thin vertical lines) show where one domain ends
and another begins.

In [ ]:
highlights = np.stack([poison, specialist])

rc = RankCarpet(
    log_scale=True,
    highlight_labels=["poison (uniform)", "specialist (medicine)"],
    reference_ranks=[10, 100],
    sort_by=0,
)
rc.fit(
    queries=queries,
    corpus=corpus,
    highlight=highlights,
    query_labels=query_labels,
)
fig = rc.plot()
fig.savefig("rank_carpet.png", dpi=300, bbox_inches="tight")
fig

## 3. Similarity Waterfall

The Similarity Waterfall shows the actual cosine similarity of each
highlight document per query, overlaid with the top-k threshold.
Green shading indicates the highlight is above the threshold;
red shading indicates it falls below.

Notice the poison clears the top-10 threshold across nearly all
queries, while the specialist only clears it in its home domain.

In [ ]:
sw = SimilarityWaterfall(
    k=10,
    highlight_labels=["poison (uniform)", "specialist (medicine)"],
    shade_margin=True,
)
sw.fit(
    queries=queries,
    corpus=corpus,
    highlight=highlights,
)
fig = sw.plot()
fig.savefig("similarity_waterfall.png", dpi=300, bbox_inches="tight")
fig

## 4. Rank Distribution

The Rank Distribution shows how each highlight document's rank is
distributed across queries.  We show three modes:

- **Histogram** — raw rank frequencies.
- **CDF** — cumulative proportion of queries where rank <= x.
- **Faceted histogram** — one panel per domain.

In [ ]:
# CDF mode — the clearest summary.
rd = RankDistribution(
    mode="cdf",
    highlight_labels=["poison (uniform)", "specialist (medicine)"],
    reference_ranks=[10, 100],
)
rd.fit(
    queries=queries,
    corpus=corpus,
    highlight=highlights,
)
fig = rd.plot()
fig.savefig("rank_distribution_cdf.png", dpi=300, bbox_inches="tight")
fig

In [ ]:
# Faceted histogram — per-domain breakdown.
rd_faceted = RankDistribution(
    mode="histogram",
    facet_by_label=True,
    highlight_labels=["poison (uniform)", "specialist (medicine)"],
)
rd_faceted.fit(
    queries=queries,
    corpus=corpus,
    highlight=highlights,
    query_labels=query_labels,
)
fig = rd_faceted.plot()
fig.savefig("rank_distribution_faceted.png", dpi=300, bbox_inches="tight")
fig

## 5. Using the precomputed similarity path

If you already have a similarity matrix (e.g. from a cross-encoder or
BM25), skip the embedding computation entirely:

In [ ]:
# Precompute similarities ourselves, then pass them in.
S_corpus = queries @ corpus.T
S_highlight = queries @ highlights.T

rc_precomp = RankCarpet(
    highlight_labels=["poison", "specialist"],
)
rc_precomp.fit(
    similarities=S_corpus,
    highlight_similarities=S_highlight,
)
fig = rc_precomp.plot()
fig

## 6. quick_plot — one-liner convenience

For fast exploration, `quick_plot` wraps the create-fit-plot cycle:

In [ ]:
from rankviz import quick_plot

fig = quick_plot(
    queries=queries,
    corpus=corpus,
    highlight=poison,
    kind="rank_carpet",
    log_scale=True,
)
fig

## Key takeaway

The poison document has **moderate** cosine similarity to every query
cluster, while each specialist has **high** similarity to its home
cluster and **low** similarity elsewhere. Yet the poison consistently
lands in the top-10 across all 150 queries, while each specialist only
ranks well in its own 50-query domain.

No standard dimensionality-reduction plot can show this: in 2-D
projections, the specialists appear "closer" to their home queries
than the poison does. rankviz reveals the retrieval truth by plotting
rank and similarity directly — no lossy projection required.